In [5]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
import pandas as pd
import lazyqsar
import numpy as np
import os

st = "alphafold2_P9WFS9_model_0_pocket_1"
perc = "bin_1"

root = '.'
# root = os.path.dirname(os.path.abspath(__file__))

# Get input data
PATH_TO_REPORTS = os.path.join(root, "..", "processed", "unidock_docking", "binarized_reports")
PATH_TO_OUTPUT = os.path.join(root, "..", "processed", "unidock_docking", "models", st, perc)
PATH_TO_EMBEDDINGS = os.path.join(root, "..", "processed", "enamine_characterization")
os.makedirs(PATH_TO_OUTPUT, exist_ok=True)

# Load compounds and activities
report = pd.read_csv(os.path.join(PATH_TO_REPORTS, f"report_bin_{st}.csv"))
compounds = report["compound"].tolist()
Y = np.array(report[perc].tolist())

# Load ids and embeddings
ids = open(os.path.join(PATH_TO_EMBEDDINGS, "IDs_CheMeleon.txt")).read().splitlines()
embeddings = np.load(os.path.join(PATH_TO_EMBEDDINGS, "X_CheMeleon.npz"))['X']

# Mapping id to embedding
id_to_embedding = {i: j for i,j in zip(ids, embeddings)}

# Creating matrix
X = np.array([id_to_embedding[i] for i in compounds])

# Stratified 5-fold CV
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
aucs = []

In [6]:
import random

random.seed(42)

sample_indices = random.sample(range(0, 100000), 5000)

X, Y = X[sample_indices], Y[sample_indices]

In [7]:
for train_idx, test_idx in kf.split(X, Y):

    # Train test split
    X_train, X_test = X[train_idx], X[test_idx]
    Y_train, Y_test = Y[train_idx], Y[test_idx]

    # Train model only on training set
    model = lazyqsar.LazyBinaryClassifier(model_type="random_forest", pca=False, min_seen_across_partitions=1, 
                                          num_trials=20, base_num_splits=1, max_samples=10000)
    model.fit(X_train, Y_train)
    probs = model.predict_proba(X_test)[:, 1]
    
    auc = roc_auc_score(Y_test, probs)
    aucs.append(auc)

    break

print(f"Mean AUROC: {np.mean(aucs):.3f} ± {np.std(aucs):.3f}")

Total samples: 4000, positive samples: 35, negative samples: 3965
Maximum samples per partition: 10000, minimum samples per partition: 30
Positive proportion: 0.01
Original positive samples: 35, total samples: 4000
Maximum samples: 10000
Sampling 35 positive and 3465 negative samples from 3500 total samples.


  0%|          | 3/1000 [00:00<00:17, 56.13it/s]


All indices seen at least 1 times. Stopping sampling.
Unique sampled indices matrix shape: (4, 3500)


100%|██████████| 4/4 [00:18<00:00,  4.53s/it]
INFO:flaml.default.suggest:metafeature distance: 2.315953235007141
[I 2025-07-24 13:49:22,297] A new study created in memory with name: no-name-625aa724-10ad-4802-9a1d-f248b5720dd2


Indices matrix shape after redundancy removal: (2, 3500)
Original positive negative balance: positive 35, negative 3965
Avg positive samples: 35.0, avg negative samples: 3465.0
Fitting model on 3500 samples, positive samples: 35, negative samples: 3465, number of features 1937
Suggested zero-shot hyperparameters: {'n_estimators': 501, 'max_features': 0.24484242524861066, 'criterion': 'entropy', 'max_leaf_nodes': 1156, 'random_state': 12032022, 'verbose': 0, 'class_weight': 'balanced_subsample'}
Fitting...


...

Early stopping: No significant improvement in the last 5 trials.
Skipping trial due to early stopping criteria.
Skipping trial due to early stopping criteria.
Skipping trial due to early stopping criteria.
Skipping trial due to early stopping criteria.
Skipping trial due to early stopping criteria.
Skipping trial due to early stopping criteria.
Skipping trial due to early stopping criteria.
Skipping trial due to early stopping criteria.
Skipping trial due to early stopping criteria.
Skipping trial due to early stopping criteria.
Skipping trial due to early stopping criteria.
Skipping trial due to early stopping criteria.
Skipping trial due to early stopping criteria.
Best hyperparameters: {'n_estimators': 476, 'max_features': 0.4253823424614421, 'max_leaf_nodes': 1750, 'criterion': 'entropy', 'n_jobs': 8, 'random_state': 42, 'class_weight': 'balanced_subsample'}, Inner hyperparameter AUROC: 0.8776623043366691
Internal AUROC CV-0: 0.9318065178342314
Logistic regression for calibration..

Predicting chunks...: 1it [00:00,  8.37it/s]


[VarianceThreshold(threshold=0)] BaseRandomForestBinaryClassifier(num_splits=1, num_trials=20, timeout=120)


Predicting chunks...: 1it [00:00,  6.01it/s]

Mean AUROC: 0.969 ± 0.000


In [ ]:
probs = model.predict_proba(X_test)[:, 1]
roc_auc_score(Y_test, probs)

In [ ]:
probs = model.predict_proba(X_train)[:, 1]
roc_auc_score(Y_train, probs)